# SuperCodeur LLM — Kaggle Training
**100M–350M param transformer from scratch on T4 ×2**

### Setup
1. **Clone repo** (automatic in cell 1)
2. **Connect Internet** : Settings → Internet = ON
3. **Enable GPU** : Settings → Accelerator = GPU T4 ×2
4. **Run All**

Checkpoints saved in `/kaggle/working/checkpoints/` — download after training.

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/ABDOULAYE-HAMIDO/Maigacode.git'
REPO_DIR = '/kaggle/working/Maigacode'

if not os.path.exists(REPO_DIR):
    !git clone "{REPO_URL}" "{REPO_DIR}"
else:
    !git -C "{REPO_DIR}" pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

print(f'Working dir: {os.getcwd()}')
print(f'Files: {len(os.listdir("."))}')

In [ ]:
import torch

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {props.name} | VRAM: {props.total_mem/1e9:.1f} GB | Compute: {props.major}.{props.minor}')

print(f'torch.compile available: {hasattr(torch, "compile")}')
print(f'BF16 support: {torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}')

In [ ]:
# Download code dataset from HuggingFace
DATA_DIR = f'{REPO_DIR}/data/raw_big'
os.makedirs(DATA_DIR, exist_ok=True)

existing = len(os.listdir(DATA_DIR)) if os.path.isdir(DATA_DIR) else 0
if existing < 100:
    print('Downloading dataset (~58K code files)...')
    !python "{REPO_DIR}/download_code.py" --output "{DATA_DIR}" --max_files 50000
else:
    print(f'Data already exists: {existing} files')

In [ ]:
# Tokenizer + dataset preparation
from model.data.prepare import prepare_data

SEQ_LEN = 1024
MAX_FILES = 50000
TOKENIZER_PATH = f'{REPO_DIR}/checkpoints/tokenizer'
DATA_PATH = f'{REPO_DIR}/checkpoints/data'

if not os.path.exists(f'{DATA_PATH}/train.pt'):
    prepare_data(
        input_dir=DATA_DIR,
        output_path=DATA_PATH,
        tokenizer_path=TOKENIZER_PATH,
        vocab_size=32000,
        max_files=MAX_FILES,
        seq_len=SEQ_LEN,
    )
else:
    print('Dataset already prepared')

In [ ]:
# ═══════════════════════════════════════════
# CONFIGURATION — Edit this cell
# ═══════════════════════════════════════════

CONFIG_NAME = '300M'  # nano | 10M | 50M | 100M | 300M | 350M
EPOCHS = 5
BATCH_SIZE = 8         # T4: 8 for 300M, 16 for 100M
LEARNING_RATE = 3e-4

# ── VRAM estimates for 300M @ batch=8, seq=1024 ─────────
# Weights (fp16):        ~516 MB
# Gradients (fp32):      ~1.0 GB
# Optimizer (AdamW):     ~2.0 GB
# Activations (SDPA):    ~1.5 GB
# ─────────────────────────────────────────────────────
# Total: ~5 GB / 16 GB (per T4) — fits with margin

print(f'Config: {CONFIG_NAME} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | Seq: {SEQ_LEN}')
print(f'Effective tokens/step: {BATCH_SIZE * SEQ_LEN:,}')

In [ ]:
import json, time
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from model.model.config import ModelConfig
from model.model.model import SuperCodeurModel
from model.tokenizer import BPE
from model.training.trainer import Trainer

device = 'cuda'

# Load tokenizer
tokenizer = BPE()
tokenizer.load(TOKENIZER_PATH)
vocab_size = tokenizer.vocab_size()
print(f'Vocab size: {vocab_size}')

# Model (with compilation)
config = ModelConfig.from_preset(CONFIG_NAME, vocab_size=vocab_size)
config.max_seq_len = SEQ_LEN
print(config)

model = SuperCodeurModel(config).to(device)
print(f'Params: {model.count_params():,}')

# torch.compile — works on T4 (compute 7.5, Triton ready)
model = torch.compile(model)
print('torch.compile enabled')

# Load data
train_data = torch.load(f'{DATA_PATH}/train.pt')
val_data = torch.load(f'{DATA_PATH}/val.pt')
print(f'Train: {len(train_data)} seq, Val: {len(val_data)} seq')

# DataLoader with pin_memory for GPU
train_loader = DataLoader(
    train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    pin_memory=True, num_workers=4,
)
val_loader = DataLoader(
    val_data, batch_size=BATCH_SIZE, shuffle=False,
    pin_memory=True, num_workers=4,
)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

# Trainer with fp16 (T4 doesn't support bf16)
grad_accum = max(1, 512 // BATCH_SIZE)
max_steps = len(train_loader) * EPOCHS // grad_accum

trainer = Trainer(
    model=model, device=device,
    learning_rate=LEARNING_RATE, weight_decay=0.1, grad_clip=1.0,
    grad_accum_steps=grad_accum,
    warmup_steps=min(200, max_steps // 10),
    max_steps=max_steps,
    betas=(0.9, 0.95),
    precision='fp16',  # T4 → fp16 (bf16 unsupported)
)

print(f'\n=== Training {CONFIG_NAME} on {CONFIG_NAME} ===')
print(f'  Grad accum: {grad_accum}, Total steps: {max_steps}')
print(f'  Effective batch: {BATCH_SIZE * grad_accum} seq/step')

t0 = time.time()
trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=EPOCHS,
    log_interval=max(1, len(train_loader) // 10),
    val_interval=max(1, len(train_loader)),
    save_dir=f'{REPO_DIR}/checkpoints',
)
elapsed = time.time() - t0
print(f'\nTraining time: {elapsed/60:.1f} min ({elapsed/3600:.2f}h)')

In [ ]:
# Test generation
best = f'{REPO_DIR}/checkpoints/best_model.pt'
if os.path.exists(best):
    # Re-load without compile (compile is for training, not inference here)
    model = SuperCodeurModel(config).to(device)
    model.load_state_dict(torch.load(best, map_location=device)['model_state_dict'])
    model = torch.compile(model)

    for prompt in ['def fibonacci', 'def add', 'class Stack', 'function hello', 'import numpy']:
        inp = torch.tensor([tokenizer.encode(prompt)], device=device)
        out = model.generate(inp, max_new_tokens=80, temperature=0.6, top_k=40, top_p=0.9,
                            repetition_penalty=1.15, eos_token_id=2)
        print(f'\n>> {prompt}')
        print(tokenizer.decode(out[0].tolist())[:500])

In [ ]:
# Save final model + package for download
import zipfile

OUTPUT_ZIP = '/kaggle/working/supercodeur_checkpoint.zip'
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in Path(f'{REPO_DIR}/checkpoints').rglob('*'):
        if f.is_file():
            zf.write(f, f.relative_to(REPO_DIR))

print(f'Checkpoint saved: {OUTPUT_ZIP}')
print(f'Size: {os.path.getsize(OUTPUT_ZIP)/1024**2:.1f} MB')
print('\nDownload from Kaggle → Output tab → Download all')